In [7]:
import duckdb
con = duckdb.connect("../data/olist.duckdb", read_only=True)
con.sql("SHOW TABLES").df()

,name
0,category_translation
1,customers
2,order_items
3,orders
4,payments
5,products
6,reviews
7,sellers


In [8]:
for table in con.sql("SHOW TABLES").df()["name"]:
    print(f"\n===== {table} =====")
    display(con.sql(f"DESCRIBE {table}").df()[["column_name", "column_type"]])
    


===== category_translation =====


,column_name,column_type
0,product_category_name,VARCHAR
1,product_category_name_english,VARCHAR



===== customers =====


,column_name,column_type
0,customer_id,VARCHAR
1,customer_unique_id,VARCHAR
2,customer_zip_code_prefix,VARCHAR
3,customer_city,VARCHAR
4,customer_state,VARCHAR



===== order_items =====


...

,column_name,column_type
0,product_id,VARCHAR
1,product_category_name,VARCHAR
2,product_name_lenght,BIGINT
3,product_description_lenght,BIGINT
4,product_photos_qty,BIGINT
5,product_weight_g,BIGINT
6,product_length_cm,BIGINT
7,product_height_cm,BIGINT
8,product_width_cm,BIGINT



===== reviews =====


,column_name,column_type
0,review_id,VARCHAR
1,order_id,VARCHAR
2,review_score,BIGINT
3,review_comment_title,VARCHAR
4,review_comment_message,VARCHAR
5,review_creation_date,TIMESTAMP
6,review_answer_timestamp,TIMESTAMP



===== sellers =====


,column_name,column_type
0,seller_id,VARCHAR
1,seller_zip_code_prefix,VARCHAR
2,seller_city,VARCHAR
3,seller_state,VARCHAR


In [9]:
con.sql("""
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp) AS mois,
        COUNT(DISTINCT o.order_id)                     AS nb_commandes,
        ROUND(SUM(i.price), 0)                         AS chiffre_affaires
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY mois
    ORDER BY mois
""").df()

,mois,nb_commandes,chiffre_affaires
0,2016-09-01,1,135.0
1,2016-10-01,265,40325.0
2,2016-12-01,1,11.0
3,2017-01-01,750,111798.0
4,2017-02-01,1653,234223.0
5,2017-03-01,2546,359199.0
6,2017-04-01,2303,340670.0
7,2017-05-01,3546,489338.0
8,2017-06-01,3135,421923.0
9,2017-07-01,3872,481605.0


In [ ]:
verifs = {
    "Produits sans catégorie":
        "SELECT COUNT(*) FROM products WHERE product_category_name IS NULL",
    "Catégories sans traduction":
        """SELECT DISTINCT p.product_category_name
           FROM products p
           LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
           WHERE t.product_category_name IS NULL AND p.product_category_name IS NOT NULL""",
    "Commandes avec plusieurs paiements":
        "SELECT COUNT(*) FROM (SELECT order_id FROM payments GROUP BY order_id HAVING COUNT(*) > 1)",
    "Types de paiement":
        "SELECT payment_type, COUNT(*) AS nb FROM payments GROUP BY payment_type ORDER BY nb DESC",
    "Répartition des notes":
        "SELECT review_score, COUNT(*) AS nb FROM reviews GROUP BY review_score ORDER BY review_score",
    "Commandes avec plusieurs avis":
        "SELECT COUNT(*) FROM (SELECT order_id FROM reviews GROUP BY order_id HAVING COUNT(*) > 1)",
    "Type de la colonne code postal":
        "SELECT column_name, data_type FROM information_schema.columns WHERE column_name = 'customer_zip_code_prefix'",
}

for titre, requete in verifs.items():
    print(f"\n===== {titre} =====")
    display(con.sql(requete).df())